In [ ]:
## Agent inference and memory update
                               
                    output = {}
                    if(memories[agent_i] is not None):
                        model.load_memories(memories[agent_i])
                    else:
                        model.reset_memories()

                    with torch.no_grad():
                        x, _ = model(batch[agent_i:agent_i+1], queries = model.queries)
                        output['audio'] = rearrange(model.projections['audio'](
                            x[:, :audio_future, :]), "b t t1 -> b (t t1)" )
                    start_modality = audio_future
                    for part, params in cf.body_outputs.items():
                        output[part] = model.projections[part](
                            x[:, start_modality:start_modality+frames_future, :] )
                        start_modality += frames_future
                    output['actions'] = torch.normal(torch.tanh(output['actions_mean']), 
                                                     1+torch.tanh(output['actions_std'])/2).detach()
                    output['actions'] = torch.clamp(output['actions'], min=-1, max=1)
                    new_actions.append(rearrange(output['actions'], '() t d -> t d'))
                    memories[agent_i] = model.return_memories(device=device)

                    # Compute loss

                    if replay_training:
                        # !!!!!!!! Warning, delete 0 later !!!!!!
                        loss = aux_loss * 0
                        start_modality = audio_future
                        # loss = loss + loss_flow*F.l1_loss(output['audio'], output_training['audio'])
                        for part, params in cf.body_outputs.items():
                            pass
                            # loss = loss + F.l1_loss(output_training[part], output[part].detach())
                        # Objective log likelihood: log πθ(a | s, g) = -0.5 * log(2πσ^2) - ((a - μ)^2) / (2σ^2)
                        variance=torch.square(1+torch.tanh(output_training['actions_std'])/2)
                        mean=torch.tanh(output_training['actions_mean'])
                        log_likelihood = 0.5 * torch.log(2*np.pi*variance) + torch.square(output['actions'] \
                                         - mean) / (eps+2*variance)
                        # Entropy: H = 0.5 * log(2π * e * σ^2)
                        entropy = 0.5 * torch.log(2*np.pi*variance)
                        loss = loss + torch.sum(log_likelihood) - c_entropy*torch.sum(entropy)
                        loss /= (numEnvs*accumulation_steps)
                        loss.backward()
                        backward_count += 1